# 02 — Entrenamiento K-Means Clustering de Clientes
Segmentación en 5 grupos: Comprador premium, Arrendatario joven,
Inversor patrimonial, Familia en crecimiento, Profesional soltero


In [ ]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

CSV_PATH   = '../ml_models/datasets/ML_NoSupervisado_Clustering_Clientes.csv'
MODEL_PATH = '../ml_models/trained/kmeans_clients_v1.pkl'
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)


In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────
le_tipo = LabelEncoder()
le_zona = LabelEncoder()
df['tipo_enc'] = le_tipo.fit_transform(df['tipo_propiedad_buscada'].fillna('Departamento'))
df['zona_enc'] = le_zona.fit_transform(df['zona_preferida_1'].fillna(''))

FEATURES = ['presupuesto_max_usd','tipo_enc','habitaciones_minimo',
            'zona_enc','propiedades_vistas','contactos_realizados']

X = df[FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Features shape:', X_scaled.shape)


In [ ]:
# ── Elbow method ─────────────────────────────────────────────────────────
inertias = []
silhouettes = []
Ks = range(2, 10)
for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(Ks, inertias, 'bo-'); axes[0].set_title('Elbow')
axes[1].plot(Ks, silhouettes, 'ro-'); axes[1].set_title('Silhouette')
plt.tight_layout(); plt.show()


In [ ]:
# ── Entrenar con K=5 ─────────────────────────────────────────────────────
mlflow.set_experiment('kmeans_clientes')
with mlflow.start_run(run_name='kmeans_v1'):
    K = 5
    km = KMeans(n_clusters=K, random_state=42, n_init=20, max_iter=500)
    km.fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_)
    mlflow.log_param('n_clusters', K)
    mlflow.log_metric('silhouette', sil)
    print(f'Silhouette score: {sil:.4f}')

df['segmento'] = km.labels_
print(df.groupby('segmento')['presupuesto_max_usd'].agg(['mean','count']))


In [ ]:
# ── Guardar modelo + scaler ───────────────────────────────────────────────
Path('../ml_models/trained').mkdir(parents=True, exist_ok=True)
# Guardamos el pipeline completo como objeto compuesto
import pickle
bundle = {'kmeans': km, 'scaler': scaler,
          'le_tipo': le_tipo, 'le_zona': le_zona, 'features': FEATURES}
joblib.dump(bundle, MODEL_PATH)
print(f'Modelo guardado en {MODEL_PATH}')

# Visualización PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)
plt.figure(figsize=(8,6))
scatter = plt.scatter(X_2d[:,0], X_2d[:,1], c=df['segmento'], cmap='tab10', alpha=0.5)
plt.colorbar(scatter)
plt.title('Segmentos de Clientes (PCA 2D)')
plt.show()
